# Data Loading and First Look

Before we train anything, score anything, or stress-test anything, we need to actually look at the data. This notebook loads HH-RLHF, flattens it into a usable shape, and asks the most basic questions: how big is it, what does a row look like, and are chosen responses actually different from rejected ones?

First run downloads the dataset (~1 GB). Every run after that loads from local cache.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from inference_lens.data.loader import load_hh_rlhf, flatten_hh_rlhf, split_dataframe
from inference_lens.utils.logging import setup_logging

setup_logging()
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_colwidth', 120)

## Load HH-RLHF

Pulling both subsets from HuggingFace: helpful-base (responses that are actually useful) and harmless-base (responses that don't cause harm). Together they cover the two things Anthropic cared about when collecting this data.

In [ ]:
datasets = load_hh_rlhf(cache_dir='../../data/raw')

for subset, ds in datasets.items():
    print(f'{subset}: {ds}')

## Flatten to a single DataFrame

HuggingFace gives us a nested DatasetDict. We flatten it so each row is one preference pair: a chosen response (what a human preferred) and a rejected one (what they didn't).

In [ ]:
df = flatten_hh_rlhf(datasets)

print(f'Total rows: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## How is the data split across subsets?

In [ ]:
print('By subset:')
print(df['subset'].value_counts())
print()
print('By original split:')
print(df['original_split'].value_counts())

## Read one example

Numbers are fine, but nothing beats reading an actual row. Let's see what a chosen vs rejected pair looks like in the wild.

In [ ]:
sample = df.sample(1, random_state=42).iloc[0]

print('Subset:', sample['subset'])
print('=' * 80)
print('CHOSEN (preferred):')
print(sample['chosen'])
print()
print('REJECTED:')
print(sample['rejected'])

## Response length distributions

Do humans prefer longer responses, or is it more nuanced than that? This is the first hint at what our features will need to capture.

In [ ]:
# 5000 rows is enough to see the shape without waiting around
sample_df = df.sample(5000, random_state=42).copy()
sample_df['chosen_len'] = sample_df['chosen'].str.split().str.len()
sample_df['rejected_len'] = sample_df['rejected'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sample_df['chosen_len'], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('Chosen response length (tokens)')
axes[0].set_xlabel('Token count')
axes[0].set_ylabel('Frequency')

axes[1].hist(sample_df['rejected_len'], bins=60, color='tomato', alpha=0.8, edgecolor='white')
axes[1].set_title('Rejected response length (tokens)')
axes[1].set_xlabel('Token count')

plt.suptitle('HH-RLHF response lengths (5K sample)')
plt.tight_layout()
plt.savefig('../../reports/figures/01_response_length_dist.png', dpi=150, bbox_inches='tight')
plt.show()

for label, col in [('Chosen  ', 'chosen_len'), ('Rejected', 'rejected_len')]:
    s = sample_df[col]
    print(f'{label}  mean: {s.mean():.1f}   median: {s.median():.1f}   max: {s.max()}')

## Train / val / test split

70/15/15 with a fixed seed. Every experiment in this project sees the exact same split.

In [ ]:
splits = split_dataframe(df, train=0.70, val=0.15, test=0.15, seed=42)

for name, split in splits.items():
    print(f'{name:6s}: {len(split):>7,} rows  ({len(split)/len(df)*100:.1f}%)')

## Save to disk

Cache the flattened DataFrame so the next notebook skips straight to features without re-flattening 170K rows.

In [ ]:
import os
os.makedirs('../../data/processed', exist_ok=True)

df.to_parquet('../../data/processed/hh_rlhf_flat.parquet', index=False)
print(f'Saved {len(df):,} rows to data/processed/hh_rlhf_flat.parquet')